# Claude vs modelo combinado

Evalua las predicciones de `data/resultados_claude_code.csv` contra el ground truth de `data/pr_datasets_filtered.csv` y contra el modelo combinado ya guardado en `mlruns`.

Este notebook no entrena el XGBoost y no trackea nada en MLflow. Solo carga el modelo combinado existente y calcula metricas.

In [1]:
from __future__ import annotations

import hashlib
import sqlite3
from pathlib import Path
from urllib.parse import urlparse, unquote

import joblib
import mlflow.xgboost
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'resultados_claude_code.csv').exists():
    ROOT = ROOT.parent

CLAUDE_CSV = ROOT / 'data' / 'resultados_claude_code.csv'
BASE_CSV = ROOT / 'data' / 'pr_datasets_filtered.csv'
SEMANTIC_CSV = ROOT / 'data' / 'pr_features.csv'
MLFLOW_DB = ROOT / 'mlflow.db'
EXPERIMENT_NAME = 'pr_feature_ablation'
MODEL_ARTIFACT_RELATIVE = Path('models') / 'model_combined'

for path in [CLAUDE_CSV, BASE_CSV, SEMANTIC_CSV, MLFLOW_DB]:
    assert path.exists(), f'No existe {path}'

print({'root': str(ROOT), 'sklearn': sklearn.__version__})

{'root': 'C:\\Users\\nicok\\Downloads\\tp-producto-nlp', 'sklearn': '1.9.0'}


## Cargar predicciones de Claude y ground truth

In [2]:
def normalize_claude_response(value: object) -> int:
    text = str(value).strip().lower()
    if text in {'no mergear', 'no_mergear', 'no merge', 'no-merge', 'rechazar', 'reject'}:
        return 0
    if text in {'mergear', 'merge', 'aceptar', 'accept'}:
        return 1
    raise ValueError(f'Respuesta de Claude no reconocida: {value!r}')

claude = pd.read_csv(CLAUDE_CSV).rename(columns={'pr': 'pr_url'})
assert {'pr_url', 'respuesta'} <= set(claude.columns)
assert not claude['pr_url'].duplicated().any(), 'Hay PRs duplicadas en resultados_claude_code.csv'
claude['claude_pred'] = claude['respuesta'].map(normalize_claude_response).astype('int8')

base_labels = pd.read_csv(BASE_CSV, usecols=['pr_url', 'merge', 'closed_at'])
eval_rows = claude.merge(
    base_labels.rename(columns={'merge': 'gt'}),
    on='pr_url',
    how='left',
    validate='one_to_one',
)
missing_gt = eval_rows['gt'].isna().sum()
assert missing_gt == 0, f'{missing_gt} PRs de Claude no existen en pr_datasets_filtered.csv'
eval_rows['gt'] = eval_rows['gt'].astype('int8')

print({'prs_claude': len(eval_rows), 'merge_rate_gt': eval_rows['gt'].mean()})
display(eval_rows[['pr_url', 'gt', 'respuesta', 'claude_pred', 'justificacion']].head())

{'prs_claude': 100, 'merge_rate_gt': np.float64(0.5)}


,pr_url,gt,respuesta,claude_pred,justificacion
0,https://api.github.com/repos/pallets/flask/pul...,0,no mergear,0,El contenido tecnico de fondo (usar una lista ...
1,https://api.github.com/repos/netdata/netdata/p...,0,no mergear,0,El diff introduce dos llamadas a lstat() cuand...
2,https://api.github.com/repos/moment/moment/pul...,0,mergear,1,"El cambio es minimo, autocontenido y corrige u..."
3,https://api.github.com/repos/jquery/jquery/pul...,1,mergear,1,El commit corrige una regresion real introduci...
4,https://api.github.com/repos/h5bp/html5-boiler...,1,no mergear,0,El commit solo revierte una linea de .gitignor...


## Encontrar y cargar el modelo combinado desde MLflow

In [3]:
def artifact_uri_to_path(uri: str) -> Path:
    parsed = urlparse(uri)
    if parsed.scheme == 'file':
        raw_path = unquote(parsed.path)
        if parsed.netloc:
            raw_path = f'//{parsed.netloc}{raw_path}'
        if raw_path.startswith('/') and len(raw_path) > 2 and raw_path[2] == ':':
            raw_path = raw_path[1:]
        return Path(raw_path)
    return Path(uri)

with sqlite3.connect(MLFLOW_DB) as con:
    run = con.execute(
        """
        select r.run_uuid, r.artifact_uri, r.start_time
        from runs r
        join experiments e on r.experiment_id = e.experiment_id
        where e.name = ? and r.status = 'FINISHED'
        order by r.start_time desc
        limit 1
        """,
        (EXPERIMENT_NAME,),
    ).fetchone()

assert run is not None, f'No hay runs FINISHED para {EXPERIMENT_NAME!r} en {MLFLOW_DB}'
run_id, artifact_uri, start_time = run
artifact_root = artifact_uri_to_path(artifact_uri)
model_dir = artifact_root / MODEL_ARTIFACT_RELATIVE
assert model_dir.exists(), f'No existe el modelo combinado en {model_dir}'

combined_model = mlflow.xgboost.load_model(str(model_dir))
print({
    'run_id': run_id,
    'artifact_root': str(artifact_root),
    'model_dir': str(model_dir),
    'model_type': type(combined_model).__name__,
})

{'run_id': 'c400c5b68c414682bce9467e8e158bb4', 'artifact_root': 'C:\\Users\\nicok\\Downloads\\tp-producto-nlp\\mlruns\\1\\c400c5b68c414682bce9467e8e158bb4\\artifacts', 'model_dir': 'C:\\Users\\nicok\\Downloads\\tp-producto-nlp\\mlruns\\1\\c400c5b68c414682bce9467e8e158bb4\\artifacts\\models\\model_combined', 'model_type': 'XGBClassifier'}


## Reconstruir features y preprocessor del split original

El run actual guarda el XGBoost combinado, pero no guarda el `ColumnTransformer`. Para alimentar el modelo con las mismas columnas, se reconstruye el preprocessor determin?sticamente sobre el split de train original. Esto no reentrena el modelo.

In [4]:
BASE_EXCLUSIONS = {
    'pr_url': 'join key',
    'merge': 'label',
    'github_merge': 'label component',
    'comment_merge': 'label component',
    'commit_merge': 'label component',
    'closed_at': 'post prediction timestamp',
    'login': 'high cardinality id',
    'body': 'raw text',
    'title': 'raw text',
    'commit_message': 'raw text',
    'code_patch_diff': 'raw diff',
}
SEQUENCE_COLUMNS = [
    'everyday_pr_comment_count_in_lifetime',
    'everyday_pr_commit_count_in_lifetime',
]
SEMANTIC_EXCLUSIONS = {
    'pr_url': 'join key',
    'merge': 'label',
    'repo': 'repo id',
    'pr_number': 'pr id / temporal proxy',
    'bloque': 'operational control',
    'elapsed_seconds': 'extraction runtime',
}
VALIDATION_SIZE = 0.10


def sequence_summaries(series: pd.Series, prefix: str) -> pd.DataFrame:
    def parse(value: object) -> np.ndarray:
        if pd.isna(value) or str(value).strip() == '':
            return np.array([], dtype=float)
        return np.asarray([float(item) for item in str(value).split(';')], dtype=float)

    arrays = series.map(parse)
    return pd.DataFrame({
        f'{prefix}__sum': arrays.map(lambda x: float(x.sum()) if x.size else np.nan),
        f'{prefix}__mean': arrays.map(lambda x: float(x.mean()) if x.size else np.nan),
        f'{prefix}__max': arrays.map(lambda x: float(x.max()) if x.size else np.nan),
        f'{prefix}__nonzero_periods': arrays.map(lambda x: int(np.count_nonzero(x)) if x.size else np.nan),
    }, index=series.index)


def coerce_model_dtypes(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in result.columns:
        if pd.api.types.is_bool_dtype(result[column]):
            result[column] = result[column].astype('int8')
        elif not pd.api.types.is_numeric_dtype(result[column]):
            parsed = pd.to_numeric(result[column], errors='coerce')
            if parsed.notna().sum() == result[column].notna().sum():
                result[column] = parsed
    return result


def drop_uninformative(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str], dict[str, str]]:
    constants = [c for c in frame if frame[c].nunique(dropna=False) <= 1]
    reduced = frame.drop(columns=constants)
    fingerprints: dict[tuple[str, str], str] = {}
    duplicates: dict[str, str] = {}
    for column in reduced.columns:
        hashed = pd.util.hash_pandas_object(reduced[column], index=False).values.tobytes()
        key = (str(reduced[column].dtype), hashlib.sha256(hashed).hexdigest())
        prior = fingerprints.get(key)
        if prior is not None and reduced[column].equals(reduced[prior]):
            duplicates[column] = prior
        else:
            fingerprints[key] = column
    return reduced.drop(columns=list(duplicates)), constants, duplicates


def build_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    numeric_columns = frame.select_dtypes(include=np.number).columns.tolist()
    categorical_columns = [c for c in frame.columns if c not in numeric_columns]
    numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ])
    categorical_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='__MISSING__')),
        ('encoder', OneHotEncoder(
            handle_unknown='infrequent_if_exist',
            min_frequency=5,
            sparse_output=True,
        )),
    ])
    return ColumnTransformer(
        [('numeric', numeric_pipeline, numeric_columns), ('categorical', categorical_pipeline, categorical_columns)],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )

In [5]:
semantic_raw = pd.read_csv(SEMANTIC_CSV, low_memory=False)
semantic = semantic_raw.drop_duplicates('pr_url', keep='last').copy()

base_columns = pd.read_csv(BASE_CSV, nrows=0).columns.tolist()
base_predictor_columns = [c for c in base_columns if c not in BASE_EXCLUSIONS]
base_usecols = ['pr_url', 'merge', 'closed_at', *base_predictor_columns]
target_urls = set(semantic['pr_url'])

matched_chunks = []
for chunk in pd.read_csv(BASE_CSV, usecols=base_usecols, chunksize=50_000, low_memory=False):
    matched = chunk.loc[chunk['pr_url'].isin(target_urls)]
    if not matched.empty:
        matched_chunks.append(matched)
base = pd.concat(matched_chunks, ignore_index=True)

common_urls = base['pr_url'].tolist()
base_indexed = base.set_index('pr_url').loc[common_urls]
semantic_indexed = semantic.set_index('pr_url').loc[common_urls]
y = base_indexed['merge'].astype('int8')

X_base = base_indexed.drop(columns=['merge', 'closed_at']).copy()
for sequence_column in SEQUENCE_COLUMNS:
    if sequence_column in X_base:
        X_base = pd.concat(
            [X_base.drop(columns=sequence_column), sequence_summaries(X_base[sequence_column], sequence_column)],
            axis=1,
        )
X_base = coerce_model_dtypes(X_base)
X_base, base_constants, base_duplicates = drop_uninformative(X_base)

semantic_feature_columns = [c for c in semantic_indexed.columns if c not in SEMANTIC_EXCLUSIONS]
X_semantic = coerce_model_dtypes(semantic_indexed[semantic_feature_columns])
X_semantic, semantic_constants, semantic_duplicates = drop_uninformative(X_semantic)

X_combined = pd.concat([
    X_base.add_prefix('base__'),
    X_semantic.add_prefix('semantic__'),
], axis=1)

split_position = int(len(y) * (1 - VALIDATION_SIZE))
train_positions = np.arange(split_position)
validation_urls = set(X_combined.index[split_position:])

eval_positions = X_combined.index.get_indexer(eval_rows['pr_url'])
assert (eval_positions >= 0).all(), 'Hay PRs de Claude fuera del dataset usado por el modelo combinado'
eval_rows['is_original_validation'] = eval_rows['pr_url'].isin(validation_urls)

print({
    'rows_model_dataset': len(X_combined),
    'raw_combined_features': X_combined.shape[1],
    'claude_rows': len(eval_rows),
    'claude_rows_in_original_validation': int(eval_rows['is_original_validation'].sum()),
})
assert eval_rows['is_original_validation'].all(), 'No todas las PRs de Claude estan en el split de validation original' 

{'rows_model_dataset': 18934, 'raw_combined_features': 84, 'claude_rows': 100, 'claude_rows_in_original_validation': 100}


## Predecir con el modelo combinado cargado

In [6]:
preprocessor_path = model_dir / 'preprocessor.joblib'
if preprocessor_path.exists():
    preprocessor = joblib.load(preprocessor_path)
    print(f'Preprocessor cargado desde {preprocessor_path}')
else:
    preprocessor = build_preprocessor(X_combined.iloc[train_positions].copy())
    preprocessor.fit(X_combined.iloc[train_positions])
    print('Preprocessor reconstruido sobre train original; el modelo XGBoost NO se reentreno.')

X_eval = X_combined.loc[eval_rows['pr_url']]
X_eval_transformed = preprocessor.transform(X_eval)

if hasattr(combined_model, 'predict_proba'):
    combined_probability = combined_model.predict_proba(X_eval_transformed)[:, 1]
else:
    import xgboost as xgb
    combined_probability = combined_model.predict(xgb.DMatrix(X_eval_transformed))

eval_rows['combined_probability'] = combined_probability
eval_rows['combined_pred'] = (eval_rows['combined_probability'] >= 0.5).astype('int8')

display(eval_rows[[
    'pr_url', 'gt', 'claude_pred', 'combined_pred', 'combined_probability', 'respuesta', 'justificacion'
]].head())

Preprocessor reconstruido sobre train original; el modelo XGBoost NO se reentreno.


,pr_url,gt,claude_pred,combined_pred,combined_probability,respuesta,justificacion
0,https://api.github.com/repos/pallets/flask/pul...,0,0,0,0.494650,no mergear,El contenido tecnico de fondo (usar una lista ...
1,https://api.github.com/repos/netdata/netdata/p...,0,0,0,0.135896,no mergear,El diff introduce dos llamadas a lstat() cuand...
2,https://api.github.com/repos/moment/moment/pul...,0,1,1,0.571986,mergear,"El cambio es minimo, autocontenido y corrige u..."
3,https://api.github.com/repos/jquery/jquery/pul...,1,1,1,0.926360,mergear,El commit corrige una regresion real introduci...
4,https://api.github.com/repos/h5bp/html5-boiler...,1,0,1,0.924571,no mergear,El commit solo revierte una linea de .gitignor...


## Metricas Claude vs modelo combinado

In [7]:
def binary_metrics(name: str, y_true: pd.Series, y_pred: pd.Series) -> dict[str, object]:
    return {
        'source': name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_merge': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_merge': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_no_merge': precision_score(1 - y_true, 1 - y_pred, pos_label=1, zero_division=0),
        'recall_no_merge': recall_score(1 - y_true, 1 - y_pred, pos_label=1, zero_division=0),
        'confusion_matrix_[tn_fp;fn_tp]': confusion_matrix(y_true, y_pred).tolist(),
    }

metrics = pd.DataFrame([
    binary_metrics('claude_code', eval_rows['gt'], eval_rows['claude_pred']),
    binary_metrics('combined_model', eval_rows['gt'], eval_rows['combined_pred']),
])

display(metrics.style.format({
    'accuracy': '{:.4f}',
    'precision_merge': '{:.4f}',
    'recall_merge': '{:.4f}',
    'precision_no_merge': '{:.4f}',
    'recall_no_merge': '{:.4f}',
}))

comparison = eval_rows[[
    'pr_url', 'gt', 'claude_pred', 'combined_pred', 'combined_probability', 'respuesta', 'justificacion'
]].copy()
comparison['claude_correct'] = comparison['claude_pred'].eq(comparison['gt'])
comparison['combined_correct'] = comparison['combined_pred'].eq(comparison['gt'])
comparison['agreement_claude_combined'] = comparison['claude_pred'].eq(comparison['combined_pred'])

display(comparison)
comparison.to_csv(ROOT / 'experiments' / 'claude_vs_combined_comparison.csv', index=False)
metrics.to_csv(ROOT / 'experiments' / 'claude_vs_combined_metrics.csv', index=False)
print('Guardado: experiments/claude_vs_combined_comparison.csv')
print('Guardado: experiments/claude_vs_combined_metrics.csv')

,source,accuracy,precision_merge,recall_merge,precision_no_merge,recall_no_merge,confusion_matrix_[tn_fp;fn_tp]
0,claude_code,0.6900,0.6418,0.8600,0.7879,0.5200,"[[26, 24], [7, 43]]"
1,combined_model,0.7400,0.6714,0.9400,0.9000,0.5400,"[[27, 23], [3, 47]]"


,pr_url,gt,claude_pred,combined_pred,combined_probability,respuesta,justificacion,claude_correct,combined_correct,agreement_claude_combined
0,https://api.github.com/repos/pallets/flask/pul...,0,0,0,0.494650,no mergear,El contenido tecnico de fondo (usar una lista ...,True,True,True
1,https://api.github.com/repos/netdata/netdata/p...,0,0,0,0.135896,no mergear,El diff introduce dos llamadas a lstat() cuand...,True,True,True
2,https://api.github.com/repos/moment/moment/pul...,0,1,1,0.571986,mergear,"El cambio es minimo, autocontenido y corrige u...",False,False,True
3,https://api.github.com/repos/jquery/jquery/pul...,1,1,1,0.926360,mergear,El commit corrige una regresion real introduci...,True,True,True
4,https://api.github.com/repos/h5bp/html5-boiler...,1,0,1,0.924571,no mergear,El commit solo revierte una linea de .gitignor...,False,True,False
...,...,...,...,...,...,...,...,...,...,...
95,https://api.github.com/repos/sveltejs/svelte/p...,1,1,1,0.732031,mergear,"Agrega el logo y enlace de ""Bekchy"" a la misma...",True,True,True
96,https://api.github.com/repos/sveltejs/svelte/p...,0,0,1,0.725087,no mergear,El commit agrega bind: a los atributos group={...,True,False,False
97,https://api.github.com/repos/h5bp/html5-boiler...,1,1,1,0.810294,mergear,El cambio migra correctamente de gulp 3 a gulp...,True,True,True
98,https://api.github.com/repos/h5bp/html5-boiler...,1,1,1,0.973308,mergear,Es un cambio puramente de formato que reindent...,True,True,True


Guardado: experiments/claude_vs_combined_comparison.csv
Guardado: experiments/claude_vs_combined_metrics.csv
